In [6]:
# ============================================================
# HOSPITAL OPERATIONS DASHBOARD
# 20 PLOTLY VISUALIZATIONS
# JUPYTER NOTEBOOK + DASH
# ============================================================

# ------------------------------------------------------------
# 1. IMPORT LIBRARIES
# ------------------------------------------------------------

import pandas as pd
import numpy as np
import plotly.express as px
import plotly.graph_objects as go

from dash import Dash, html, dcc


# ------------------------------------------------------------
# 2. LOAD DATASET
# ------------------------------------------------------------

FILE_NAME = "Hospital_Operations_Dataset 1.csv"

df = pd.read_csv(FILE_NAME)

# Remove accidental spaces from column names
df.columns = df.columns.str.strip()

print("=" * 70)
print("DATASET LOADED SUCCESSFULLY")
print("=" * 70)

print("Rows    :", len(df))
print("Columns :", len(df.columns))

print("\nColumns:")
print(df.columns.tolist())


# ------------------------------------------------------------
# 3. REQUIRED COLUMNS
# ------------------------------------------------------------

required_columns = [
    "Patient_ID",
    "Age",
    "Gender",
    "Patient_Location",
    "Department",
    "Diagnosis",
    "Severity_Level",
    "Admit_Date",
    "Admission_Time",
    "Discharge_Time",
    "Length_of_Stay_Days",
    "Wait_Time_Minutes",
    "Doctor_ID",
    "Insurance_Type",
    "Treatment_Cost_USD",
    "Readmission_Flag",
    "Outcome"
]

missing = [
    column for column in required_columns
    if column not in df.columns
]

if missing:
    raise ValueError(
        "Missing columns in dataset: " + str(missing)
    )

print("\nAll required columns found.")


# ------------------------------------------------------------
# 4. CLEAN NUMERIC COLUMNS
# ------------------------------------------------------------

numeric_columns = [
    "Age",
    "Length_of_Stay_Days",
    "Wait_Time_Minutes",
    "Treatment_Cost_USD",
    "Readmission_Flag"
]

for column in numeric_columns:
    df[column] = pd.to_numeric(
        df[column],
        errors="coerce"
    )


# ------------------------------------------------------------
# 5. CONVERT ADMISSION DATE
# ------------------------------------------------------------

# Dataset contains dates such as:
# 02-06-2025
# 13-03-2025
#
# dayfirst=True is required because the dataset uses DD-MM-YYYY.

df["Admit_Date"] = pd.to_datetime(
    df["Admit_Date"],
    format="%d-%m-%Y",
    errors="coerce"
)

# Remove only rows with invalid admission dates
df = df.dropna(
    subset=["Admit_Date"]
).copy()

print("\nInvalid admission dates removed.")
print("Final rows:", len(df))


# ------------------------------------------------------------
# 6. CREATE DATE FEATURES
# ------------------------------------------------------------

df["Date"] = df["Admit_Date"].dt.normalize()

df["Month"] = (
    df["Admit_Date"]
    .dt.to_period("M")
    .astype(str)
)

df["Year"] = (
    df["Admit_Date"]
    .dt.year
)

df["Quarter"] = (
    "Q" +
    df["Admit_Date"]
    .dt.quarter
    .astype(str)
)

df["Day"] = (
    df["Admit_Date"]
    .dt.day_name()
)

df["Month_Name"] = (
    df["Admit_Date"]
    .dt.month_name()
)

df["Day_Number"] = (
    df["Admit_Date"]
    .dt.day
)

df["Week_Number"] = (
    df["Admit_Date"]
    .dt.isocalendar()
    .week
    .astype(int)
)


# ------------------------------------------------------------
# 7. DAILY ADMISSIONS
# ------------------------------------------------------------

daily = (
    df.groupby("Date")
      .size()
      .reset_index(name="Admissions")
      .sort_values("Date")
)

daily["Cumulative"] = (
    daily["Admissions"]
    .cumsum()
)

daily["Moving_Average_7"] = (
    daily["Admissions"]
    .rolling(
        window=7,
        min_periods=1
    )
    .mean()
)

daily["Rolling_30"] = (
    daily["Admissions"]
    .rolling(
        window=30,
        min_periods=1
    )
    .sum()
)


# ------------------------------------------------------------
# 8. MONTHLY ADMISSIONS
# ------------------------------------------------------------

monthly = (
    df.groupby("Month")
      .size()
      .reset_index(name="Admissions")
      .sort_values("Month")
)

monthly["Growth_Percent"] = (
    monthly["Admissions"]
    .pct_change()
    .fillna(0)
    * 100
)


# ------------------------------------------------------------
# 9. WEEKLY ADMISSIONS
# ------------------------------------------------------------

weekly = (
    df.groupby("Week_Number")
      .size()
      .reset_index(name="Admissions")
      .sort_values("Week_Number")
)


# ------------------------------------------------------------
# 10. WEEKDAY ADMISSIONS
# ------------------------------------------------------------

day_order = [
    "Monday",
    "Tuesday",
    "Wednesday",
    "Thursday",
    "Friday",
    "Saturday",
    "Sunday"
]

weekday = (
    df.groupby("Day")
      .size()
      .reset_index(name="Admissions")
)

weekday["Day"] = pd.Categorical(
    weekday["Day"],
    categories=day_order,
    ordered=True
)

weekday = weekday.sort_values("Day")


# ------------------------------------------------------------
# 11. QUARTERLY ADMISSIONS
# ------------------------------------------------------------

quarter = (
    df.groupby("Quarter")
      .size()
      .reset_index(name="Admissions")
)

quarter["Quarter_Number"] = (
    quarter["Quarter"]
    .str.replace("Q", "", regex=False)
    .astype(int)
)

quarter = quarter.sort_values("Quarter_Number")


# ------------------------------------------------------------
# 12. YEARLY ADMISSIONS
# ------------------------------------------------------------

yearly = (
    df.groupby("Year")
      .size()
      .reset_index(name="Admissions")
      .sort_values("Year")
)


# ------------------------------------------------------------
# 13. SEASON
# ------------------------------------------------------------

def get_season(month):

    if month in [12, 1, 2]:
        return "Winter"

    elif month in [3, 4, 5]:
        return "Summer"

    elif month in [6, 7, 8, 9]:
        return "Monsoon"

    else:
        return "Autumn"


df["Season"] = (
    df["Admit_Date"]
    .dt.month
    .apply(get_season)
)

season = (
    df.groupby("Season")
      .size()
      .reset_index(name="Admissions")
)

season_order = [
    "Winter",
    "Summer",
    "Monsoon",
    "Autumn"
]

season["Season"] = pd.Categorical(
    season["Season"],
    categories=season_order,
    ordered=True
)

season = season.sort_values("Season")


# ------------------------------------------------------------
# 14. PEAK DAYS
# ------------------------------------------------------------

peak_days = (
    daily
    .nlargest(10, "Admissions")
    .sort_values("Admissions")
)


# ------------------------------------------------------------
# 15. LOWEST DAYS
# ------------------------------------------------------------

lowest_days = (
    daily
    .nsmallest(10, "Admissions")
    .sort_values("Admissions")
)


# ------------------------------------------------------------
# 16. DEPARTMENT MONTHLY TREND
# ------------------------------------------------------------

department_month = (
    df.groupby(
        ["Month", "Department"]
    )
    .size()
    .reset_index(name="Admissions")
)


# ------------------------------------------------------------
# 17. SEVERITY MONTHLY TREND
# ------------------------------------------------------------

severity_month = (
    df.groupby(
        ["Month", "Severity_Level"]
    )
    .size()
    .reset_index(name="Admissions")
)


# ------------------------------------------------------------
# 18. READMISSION TREND
# ------------------------------------------------------------

readmission_month = (
    df.groupby(
        ["Month", "Readmission_Flag"]
    )
    .size()
    .reset_index(name="Count")
)

readmission_month["Readmission"] = (
    readmission_month["Readmission_Flag"]
    .map({
        0: "No",
        1: "Yes"
    })
    .fillna(
        readmission_month["Readmission_Flag"]
        .astype(str)
    )
)


# ============================================================
# 19. VISUALIZATION 1
# DAILY ADMISSION TREND
# ============================================================

fig_daily = px.line(
    daily,
    x="Date",
    y="Admissions",
    markers=True,
    title="1. Daily Admission Trend"
)

fig_daily.update_layout(
    template="plotly_white",
    hovermode="x unified"
)


# ============================================================
# 20. VISUALIZATION 2
# MONTHLY ADMISSION TREND
# ============================================================

fig_month = px.line(
    monthly,
    x="Month",
    y="Admissions",
    markers=True,
    title="2. Monthly Admission Trend"
)

fig_month.update_layout(
    template="plotly_white"
)


# ============================================================
# 21. VISUALIZATION 3
# WEEKLY ADMISSIONS
# ============================================================

fig_week = px.bar(
    weekly,
    x="Week_Number",
    y="Admissions",
    title="3. Weekly Admissions"
)

fig_week.update_layout(
    template="plotly_white",
    xaxis_title="Week Number"
)


# ============================================================
# 22. VISUALIZATION 4
# DAY OF WEEK
# ============================================================

fig_day = px.bar(
    weekday,
    x="Day",
    y="Admissions",
    title="4. Admissions by Day of Week"
)

fig_day.update_layout(
    template="plotly_white"
)


# ============================================================
# 23. VISUALIZATION 5
# QUARTER
# ============================================================

fig_quarter = px.bar(
    quarter,
    x="Quarter",
    y="Admissions",
    title="5. Quarter-wise Admissions"
)

fig_quarter.update_layout(
    template="plotly_white"
)


# ============================================================
# 24. VISUALIZATION 6
# YEAR
# ============================================================

fig_year = px.line(
    yearly,
    x="Year",
    y="Admissions",
    markers=True,
    title="6. Yearly Admissions"
)

fig_year.update_layout(
    template="plotly_white"
)


# ============================================================
# 25. VISUALIZATION 7
# CUMULATIVE ADMISSIONS
# ============================================================

fig_cumulative = px.line(
    daily,
    x="Date",
    y="Cumulative",
    title="7. Cumulative Admissions"
)

fig_cumulative.update_layout(
    template="plotly_white"
)


# ============================================================
# 26. VISUALIZATION 8
# 7-DAY MOVING AVERAGE
# ============================================================

fig_ma = px.line(
    daily,
    x="Date",
    y="Moving_Average_7",
    title="8. 7-Day Moving Average"
)

fig_ma.update_layout(
    template="plotly_white"
)


# ============================================================
# 27. VISUALIZATION 9
# PEAK ADMISSION DAYS
# ============================================================

fig_peak = px.bar(
    peak_days,
    x="Admissions",
    y="Date",
    orientation="h",
    text="Admissions",
    title="9. Top 10 Peak Admission Days"
)

fig_peak.update_layout(
    template="plotly_white"
)


# ============================================================
# 28. VISUALIZATION 10
# LOWEST ADMISSION DAYS
# ============================================================

fig_low = px.bar(
    lowest_days,
    x="Admissions",
    y="Date",
    orientation="h",
    text="Admissions",
    title="10. Lowest 10 Admission Days"
)

fig_low.update_layout(
    template="plotly_white"
)


# ============================================================
# 29. VISUALIZATION 11
# ADMISSION HEATMAP
# ============================================================

heatmap_data = (
    df.groupby(
        ["Month_Name", "Day_Number"]
    )
    .size()
    .reset_index(name="Admissions")
)

month_order = [
    "January",
    "February",
    "March",
    "April",
    "May",
    "June",
    "July",
    "August",
    "September",
    "October",
    "November",
    "December"
]

heatmap_data["Month_Name"] = pd.Categorical(
    heatmap_data["Month_Name"],
    categories=month_order,
    ordered=True
)

heatmap_data = heatmap_data.sort_values(
    ["Month_Name", "Day_Number"]
)

heat_matrix = heatmap_data.pivot(
    index="Month_Name",
    columns="Day_Number",
    values="Admissions"
).fillna(0)

fig_heat = px.imshow(
    heat_matrix,
    aspect="auto",
    title="11. Admission Heatmap - Month vs Day"
)

fig_heat.update_layout(
    template="plotly_white"
)


# ============================================================
# 30. VISUALIZATION 12
# ADMISSIONS BY LOCATION
# ============================================================

location = (
    df["Patient_Location"]
    .value_counts()
    .reset_index()
)

location.columns = [
    "Patient_Location",
    "Patients"
]

fig_location = px.bar(
    location.head(15),
    x="Patients",
    y="Patient_Location",
    orientation="h",
    title="12. Top Patient Locations"
)

fig_location.update_layout(
    template="plotly_white"
)


# ============================================================
# 31. VISUALIZATION 13
# DEPARTMENT DISTRIBUTION
# ============================================================

department = (
    df["Department"]
    .value_counts()
    .reset_index()
)

department.columns = [
    "Department",
    "Patients"
]

fig_department = px.bar(
    department,
    x="Department",
    y="Patients",
    title="13. Department-wise Admissions"
)

fig_department.update_layout(
    template="plotly_white"
)


# ============================================================
# 32. VISUALIZATION 14
# DIAGNOSIS DISTRIBUTION
# ============================================================

diagnosis = (
    df["Diagnosis"]
    .value_counts()
    .reset_index()
)

diagnosis.columns = [
    "Diagnosis",
    "Patients"
]

fig_diagnosis = px.bar(
    diagnosis,
    x="Patients",
    y="Diagnosis",
    orientation="h",
    title="14. Diagnosis Distribution"
)

fig_diagnosis.update_layout(
    template="plotly_white"
)


# ============================================================
# 33. VISUALIZATION 15
# SEVERITY DISTRIBUTION
# ============================================================

severity = (
    df["Severity_Level"]
    .value_counts()
    .reset_index()
)

severity.columns = [
    "Severity",
    "Patients"
]

fig_severity = px.pie(
    severity,
    names="Severity",
    values="Patients",
    title="15. Severity Distribution"
)

fig_severity.update_layout(
    template="plotly_white"
)


# ============================================================
# 34. VISUALIZATION 16
# INSURANCE DISTRIBUTION
# ============================================================

insurance = (
    df["Insurance_Type"]
    .value_counts()
    .reset_index()
)

insurance.columns = [
    "Insurance_Type",
    "Patients"
]

fig_insurance = px.pie(
    insurance,
    names="Insurance_Type",
    values="Patients",
    title="16. Insurance Type Distribution"
)

fig_insurance.update_layout(
    template="plotly_white"
)


# ============================================================
# 35. VISUALIZATION 17
# LENGTH OF STAY
# ============================================================

fig_los = px.histogram(
    df,
    x="Length_of_Stay_Days",
    nbins=20,
    title="17. Length of Stay Distribution"
)

fig_los.update_layout(
    template="plotly_white"
)


# ============================================================
# 36. VISUALIZATION 18
# WAIT TIME
# ============================================================

fig_wait = px.histogram(
    df,
    x="Wait_Time_Minutes",
    nbins=30,
    title="18. Patient Wait Time Distribution"
)

fig_wait.update_layout(
    template="plotly_white"
)


# ============================================================
# 37. VISUALIZATION 19
# TREATMENT COST
# ============================================================

fig_cost = px.histogram(
    df,
    x="Treatment_Cost_USD",
    nbins=30,
    title="19. Treatment Cost Distribution"
)

fig_cost.update_layout(
    template="plotly_white"
)


# ============================================================
# 38. VISUALIZATION 20
# OUTCOME DISTRIBUTION
# ============================================================

outcome = (
    df["Outcome"]
    .value_counts()
    .reset_index()
)

outcome.columns = [
    "Outcome",
    "Patients"
]

fig_outcome = px.pie(
    outcome,
    names="Outcome",
    values="Patients",
    title="20. Patient Outcome Distribution"
)

fig_outcome.update_layout(
    template="plotly_white"
)


# ============================================================
# 39. EXTRA ANALYSIS FIGURES
# ============================================================

# Department trend
fig_dept_trend = px.line(
    department_month,
    x="Month",
    y="Admissions",
    color="Department",
    markers=True,
    title="Department-wise Monthly Admissions"
)

fig_dept_trend.update_layout(
    template="plotly_white"
)


# Severity trend
fig_severity_trend = px.line(
    severity_month,
    x="Month",
    y="Admissions",
    color="Severity_Level",
    markers=True,
    title="Severity-wise Monthly Admissions"
)

fig_severity_trend.update_layout(
    template="plotly_white"
)


# Readmission trend
fig_readmission = px.bar(
    readmission_month,
    x="Month",
    y="Count",
    color="Readmission",
    barmode="group",
    title="Monthly Readmission Trend"
)

fig_readmission.update_layout(
    template="plotly_white"
)


# Growth
fig_growth = px.bar(
    monthly,
    x="Month",
    y="Growth_Percent",
    title="Monthly Admission Growth (%)"
)

fig_growth.update_layout(
    template="plotly_white"
)


# Rolling 30 days
fig_rolling = px.line(
    daily,
    x="Date",
    y="Rolling_30",
    title="30-Day Rolling Admissions"
)

fig_rolling.update_layout(
    template="plotly_white"
)


# ============================================================
# 40. KPI VALUES
# ============================================================

total_patients = len(df)

total_departments = df["Department"].nunique()

average_age = (
    df["Age"].mean()
)

average_wait = (
    df["Wait_Time_Minutes"].mean()
)

average_los = (
    df["Length_of_Stay_Days"].mean()
)

total_cost = (
    df["Treatment_Cost_USD"].sum()
)

average_cost = (
    df["Treatment_Cost_USD"].mean()
)

readmission_rate = (
    df["Readmission_Flag"].mean()
    * 100
)

peak_daily = (
    daily["Admissions"].max()
)

peak_date = (
    daily.loc[
        daily["Admissions"].idxmax(),
        "Date"
    ]
)


# ============================================================
# 41. PRINT SUMMARY
# ============================================================

print("\n" + "=" * 70)
print("HOSPITAL OPERATIONS SUMMARY")
print("=" * 70)

print(
    f"Total Patients       : {total_patients:,}"
)

print(
    f"Departments          : {total_departments}"
)

print(
    f"Average Age          : {average_age:.2f}"
)

print(
    f"Average Wait Time    : {average_wait:.2f} minutes"
)

print(
    f"Average Length Stay  : {average_los:.2f} days"
)

print(
    f"Average Treatment Cost: ${average_cost:,.2f}"
)

print(
    f"Total Treatment Cost : ${total_cost:,.2f}"
)

print(
    f"Readmission Rate     : {readmission_rate:.2f}%"
)

print(
    f"Peak Daily Admissions: {peak_daily}"
)

print(
    f"Peak Admission Date  : {peak_date}"
)


# ============================================================
# 42. DASH APP
# ============================================================

app = Dash(
    __name__
)


# ------------------------------------------------------------
# CARD STYLE
# ------------------------------------------------------------

card_style = {
    "backgroundColor": "#f4f6f8",
    "padding": "18px",
    "margin": "8px",
    "borderRadius": "12px",
    "boxShadow": "0 2px 8px rgba(0,0,0,0.15)",
    "display": "inline-block",
    "width": "17%",
    "minWidth": "160px",
    "textAlign": "center"
}


# ------------------------------------------------------------
# DASHBOARD LAYOUT
# ------------------------------------------------------------

app.layout = html.Div(

    [

        # ----------------------------------------------------
        # TITLE
        # ----------------------------------------------------

        html.H1(
            "🏥 Hospital Operations Dashboard",
            style={
                "textAlign": "center",
                "marginBottom": "5px"
            }
        ),

        html.P(
            "Admission Trend Analysis & Hospital Performance",
            style={
                "textAlign": "center",
                "fontSize": "18px"
            }
        ),


        # ----------------------------------------------------
        # KPI CARDS
        # ----------------------------------------------------

        html.Div(

            [

                html.Div(
                    [
                        html.H4("Total Patients"),
                        html.H2(
                            f"{total_patients:,}"
                        )
                    ],
                    style=card_style
                ),

                html.Div(
                    [
                        html.H4("Avg Wait Time"),
                        html.H2(
                            f"{average_wait:.1f} min"
                        )
                    ],
                    style=card_style
                ),

                html.Div(
                    [
                        html.H4("Avg Stay"),
                        html.H2(
                            f"{average_los:.1f} days"
                        )
                    ],
                    style=card_style
                ),

                html.Div(
                    [
                        html.H4("Readmission"),
                        html.H2(
                            f"{readmission_rate:.1f}%"
                        )
                    ],
                    style=card_style
                ),

                html.Div(
                    [
                        html.H4("Peak Daily"),
                        html.H2(
                            f"{peak_daily:,}"
                        )
                    ],
                    style=card_style
                )

            ],

            style={
                "display": "flex",
                "flexWrap": "wrap",
                "justifyContent": "center",
                "marginBottom": "25px"
            }
        ),


        # ----------------------------------------------------
        # ADMISSION TREND
        # ----------------------------------------------------

        html.H2(
            "📈 Admission Trend Analysis"
        ),

        dcc.Graph(
            id="daily-trend",
            figure=fig_daily
        ),

        dcc.Graph(
            id="monthly-trend",
            figure=fig_month
        ),

        dcc.Graph(
            id="weekly-trend",
            figure=fig_week
        ),

        dcc.Graph(
            id="weekday-trend",
            figure=fig_day
        ),

        dcc.Graph(
            id="quarter-trend",
            figure=fig_quarter
        ),

        dcc.Graph(
            id="year-trend",
            figure=fig_year
        ),

        dcc.Graph(
            id="cumulative-trend",
            figure=fig_cumulative
        ),

        dcc.Graph(
            id="moving-average",
            figure=fig_ma
        ),

        dcc.Graph(
            id="peak-days",
            figure=fig_peak
        ),

        dcc.Graph(
            id="lowest-days",
            figure=fig_low
        ),

        dcc.Graph(
            id="heatmap",
            figure=fig_heat
        ),


        # ----------------------------------------------------
        # PATIENT ANALYSIS
        # ----------------------------------------------------

        html.H2(
            "👥 Patient Analysis"
        ),

        dcc.Graph(
            id="location",
            figure=fig_location
        ),

        dcc.Graph(
            id="department",
            figure=fig_department
        ),

        dcc.Graph(
            id="diagnosis",
            figure=fig_diagnosis
        ),

        dcc.Graph(
            id="severity",
            figure=fig_severity
        ),

        dcc.Graph(
            id="insurance",
            figure=fig_insurance
        ),

        dcc.Graph(
            id="length-of-stay",
            figure=fig_los
        ),

        dcc.Graph(
            id="wait-time",
            figure=fig_wait
        ),

        dcc.Graph(
            id="treatment-cost",
            figure=fig_cost
        ),

        dcc.Graph(
            id="outcome",
            figure=fig_outcome
        ),


        # ----------------------------------------------------
        # ADVANCED ANALYSIS
        # ----------------------------------------------------

        html.H2(
            "📊 Advanced Hospital Analysis"
        ),

        dcc.Graph(
            id="department-trend",
            figure=fig_dept_trend
        ),

        dcc.Graph(
            id="severity-trend",
            figure=fig_severity_trend
        ),

        dcc.Graph(
            id="readmission-trend",
            figure=fig_readmission
        ),

        dcc.Graph(
            id="growth-trend",
            figure=fig_growth
        ),

        dcc.Graph(
            id="rolling-trend",
            figure=fig_rolling
        ),

        html.Hr(),

        html.P(
            "Hospital Operations Dashboard | Python + Pandas + Plotly + Dash",
            style={
                "textAlign": "center",
                "padding": "20px",
                "fontWeight": "bold"
            }
        )

    ],

    style={
        "fontFamily": "Arial, sans-serif",
        "padding": "20px",
        "backgroundColor": "#ffffff"
    }
)


# ============================================================
# 43. RUN DASH
# ============================================================

print("\n" + "=" * 70)
print("DASHBOARD STARTING")
print("=" * 70)

# Port 8051 is used to avoid the old 8050 process.
app.run(
    debug=False,
    port=8051
)

DATASET LOADED SUCCESSFULLY
Rows    : 100000
Columns : 17

Columns:
['Patient_ID', 'Age', 'Gender', 'Patient_Location', 'Department', 'Diagnosis', 'Severity_Level', 'Admit_Date', 'Admission_Time', 'Discharge_Time', 'Length_of_Stay_Days', 'Wait_Time_Minutes', 'Doctor_ID', 'Insurance_Type', 'Treatment_Cost_USD', 'Readmission_Flag', 'Outcome']

All required columns found.

Invalid admission dates removed.
Final rows: 100000

HOSPITAL OPERATIONS SUMMARY
Total Patients       : 100,000
Departments          : 10
Average Age          : 47.43
Average Wait Time    : 152.04 minutes
Average Length Stay  : 5.36 days
Average Treatment Cost: $4,893.89
Total Treatment Cost : $489,389,490.00
Readmission Rate     : 7.48%
Peak Daily Admissions: 330
Peak Admission Date  : 2025-12-29 00:00:00

DASHBOARD STARTING
